# DATAFEST: submissions.csv
Regresión logística con 8 variables continuas y 7 reglas, entrenada con toda la data (enero–noviembre) para predecir diciembre.

In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
print(train.shape, test.shape)

(110100, 25) (9900, 24)


In [2]:
CONTINUAS = ["dias_ultima_transaccion", "numero_productos", "ratio_deuda_ingresos", "ingresos",
             "saldo_promedio", "antiguedad_cuenta_meses", "distancia_sucursal_km", "edad"]
FLAGS = ["riesgo_low_productos_3_o_mas", "riesgo_low_movil_tarjeta_2_o_mas", "riesgo_high_transaccion_mas_180",
         "riesgo_high_canal_branch", "riesgo_high_branch_transaccion_mas_180", "movil_y_tarjeta", "transaccion_mas_180"]
VARS = CONTINUAS + FLAGS


def agregar_flags(d):
    d = d.copy()
    low = d["banda_riesgo"].eq("low")
    high = d["banda_riesgo"].eq("high")
    mt = d["activo_movil"].astype(bool) & d["tiene_tarjeta_credito"].astype(bool)
    t180 = d["dias_ultima_transaccion"] > 180
    br = d["canal_adquisicion"].eq("branch")
    d["movil_y_tarjeta"] = mt.astype(int)
    d["riesgo_low_productos_3_o_mas"] = (low & (d["numero_productos"] >= 3)).astype(int)
    d["riesgo_low_movil_tarjeta_2_o_mas"] = (low & (d["numero_productos"] >= 2) & mt).astype(int)
    d["transaccion_mas_180"] = t180.astype(int)
    d["riesgo_high_transaccion_mas_180"] = (high & t180).astype(int)
    d["riesgo_high_canal_branch"] = (high & br).astype(int)
    d["riesgo_high_branch_transaccion_mas_180"] = (high & br & t180).astype(int)
    return d


X_train = agregar_flags(train)[VARS]
X_test = agregar_flags(test)[VARS]
y_train = train["objetivo"]

In [3]:
modelo = make_pipeline(StandardScaler(), LogisticRegression(C=1e6, max_iter=5000))
modelo.fit(X_train, y_train)

submissions = pd.DataFrame({"id_cliente": test["id_cliente"], "prediccion": modelo.predict_proba(X_test)[:, 1]})
submissions.to_csv("submissions.csv", index=False)

In [4]:
assert list(submissions.columns) == ["id_cliente", "prediccion"]
assert len(submissions) == len(test) and (submissions["id_cliente"].values == test["id_cliente"].values).all()
assert submissions["prediccion"].between(0, 1).all() and submissions["prediccion"].notna().all()
print(submissions.shape)
submissions.head()

(9900, 2)


,id_cliente,prediccion
0,2,0.128301
1,3,0.054987
2,7,0.135968
3,10,0.059035
4,11,0.133056
